# NFL Weekly Analysis Notebook

This notebook loads play-by-play data, cleans it, constructs weekly outcome variables, runs exploratory analysis, and fits a linear regression model.

## Load Packages

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical modeling
import statsmodels.api as sm

# Set plotting style
sns.set(style="whitegrid")

## Load Dataset

In [ ]:
pbp = pd.read_csv("pbp_clean.csv")

# Quick look
pbp.head()

In [ ]:
pbp.shape

## Dataset Cleaning

In [ ]:
# Basic cleaning checks
pbp = pbp.dropna(subset=["season", "week", "game_id"])

# Ensure correct data types
pbp["season"] = pbp["season"].astype(int)
pbp["week"] = pbp["week"].astype(int)

pbp.head()

## Scoring Outcome Variable

In [ ]:
# Final scores per game
game_scores = (
    pbp.groupby(["season", "week", "game_id"])
    .agg({
        "home_score": "max",
        "away_score": "max"
    })
    .reset_index()
)

game_scores["total_points"] = game_scores["home_score"] + game_scores["away_score"]

# Weekly average points per game
weekly_scoring = (
    game_scores.groupby(["season", "week"])["total_points"]
    .mean()
    .reset_index(name="avg_points")
)

weekly_scoring.head()

## Pass Rate Outcome Variable

In [ ]:
pass_rate = (
    pbp[pbp["play_type"].isin(["pass", "run"])]
    .groupby(["season", "week"])
    .apply(lambda x: (x["play_type"] == "pass").mean())
    .reset_index(name="pass_rate")
)

pass_rate.head()

## Plays Per Game Outcome Variable

In [ ]:
plays_per_game = (
    pbp.groupby(["season", "week", "game_id"])
    .size()
    .reset_index(name="num_plays")
    .groupby(["season", "week"])["num_plays"]
    .mean()
    .reset_index(name="avg_plays")
)

plays_per_game.head()

## Merge All Variables

In [ ]:
merged = weekly_scoring.merge(pass_rate, on=["season", "week"]) \
                       .merge(plays_per_game, on=["season", "week"])

merged = merged.sort_values(["season", "week"])
merged["time_index"] = range(len(merged))

merged.head()

## Exploratory Data Analysis

In [ ]:
# Scoring over time
plt.figure(figsize=(12, 6))
plt.plot(merged["avg_points"])
plt.title("Weekly Average Points Per Game (2010–2024)")
plt.xlabel("Time")
plt.ylabel("Average Points")
plt.show()

In [ ]:
# Pass rate over time
plt.figure(figsize=(12, 6))
plt.plot(merged["pass_rate"])
plt.title("Weekly Pass Rate (2010–2024)")
plt.xlabel("Time")
plt.ylabel("Pass Rate")
plt.show()

In [ ]:
# Plays per game over time
plt.figure(figsize=(12, 6))
plt.plot(merged["avg_plays"])
plt.title("Weekly Plays Per Game (2010–2024)")
plt.xlabel("Time")
plt.ylabel("Average Plays")
plt.show()

In [ ]:
# Correlation matrix
merged[["avg_points", "pass_rate", "avg_plays"]].corr()

In [ ]:
# Heatmap
sns.heatmap(
    merged[["avg_points", "pass_rate", "avg_plays"]].corr(),
    annot=True,
    cmap="coolwarm"
)
plt.title("Correlation Matrix")
plt.show()

## Linear Regression

In [ ]:
X = merged[["pass_rate", "avg_plays", "time_index"]]
X = sm.add_constant(X)
y = merged["avg_points"]

model = sm.OLS(y, X).fit()
print(model.summary())

## Export Final Dataset

In [ ]:
merged.to_csv("nfl_weekly_dataset.csv", index=False)